In [1]:
import re
import google.generativeai as genai
import json
import random
import pandas as pd


random.seed(42)
with open("../data/ellipse/1st_ellipse_data.json") as file:
    data = json.load(file)


# LLM Initiation

In [2]:
with open("../api_keys.json", "r") as file:
    api_keys = json.load(file)
GOOGLE_API_KEY = api_keys["google"]

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel('gemini-1.5-pro-latest')

# Dimension Data Preparation

In [3]:
DIMENSION = "Vocabulary"

overall_rubric = '''
1 - A limited range of familiar words or phrases loosely strung together; frequent errors in grammar (including syntax) and usage. Communication impeded in most cases by language inaccuracies. 

2 - Inconsistent facility in sentence formation, word choice, and mechanics; organization partially developed but may be missing or unsuccessful. Communication impeded in many instances by language inaccuracies. 

3 - Facility limited to the use of common structures and generic vocabulary; organization generally controlled although connection sometimes absent or unsuccessful; errors in grammar and syntax and usage. Communication is impeded by language inaccuracies in some cases.

4 - Facility in the use of language with syntactic variety and range of words and phrases; controlled organization; accuracy in grammar and conventions; occasional language inaccuracies that rarely impede communication. 

5 - Native-like facility in the use of language with syntactic variety, Appropriate word choice and phrases; well-controlled text organization; precise use of grammar and conventions; rare language inaccuracies that do not impede communication. 
'''

cohesion_rubric = '''
1 - No clear control of organization; cohesive devices not present or unsuccessfully used; presentation of ideas unclear.

2 - Organization only partially developed with a lack of logical sequencing of ideas; some basic cohesive devices used but with inaccuracy or repetition. 

3 - Organization generally controlled; cohesive devices used but limited in type; Some repetitive, mechanical, or faulty use of cohesion use within and/or between sentences and paragraphs.

4 - Organization generally well controlled; a range of cohesive devices used appropriately such as reference and transitional words and phrases to connect ideas; generally appropriate overlap of ideas.

5 - Text organization consistently well controlled using a variety of effective linguistic features such as reference and transitional words and phrases to connect ideas across sentences and paragraphs; appropriate overlap of ideas.
'''

conventions_rubric = '''
1 - Minimal use of conventions; spelling, capitalization, and punctuation errors throughout.

2 - Variable use of conventions; spelling, capitalization, and punctuation errors frequent and distracting.

3 - Developing use of conventions to convey meaning; errors in spelling, capitalization, and punctuation that are sometimes distracting.

4 - Generally consistent use of appropriate conventions to convey meaning; spelling, capitalization, and punctuation errors few and not distracting.

5 - Consistent use of appropriate conventions to convey meaning; spelling, capitalization, and punctuation errors nonexistent or negligible.
'''

grammar_rubric = '''
1 - Errors in grammar and usage throughout.

2 - Many errors in grammar and usage.

3 - Some errors in grammar and usage.

4 - Minimal errors in grammar and usage.

5 - Command of grammar and usage with few or no errors.
'''

phraseology_rubric = '''
1 - Memorized chunks of language, or simple phrasal patterns predominate; many repetitions and misuses of phrases. 

2 - Narrow range of phrases, such as collocations and lexical bundles, used to convey basic and elementary meaning; many repetitions and /or misuses of phrases. 

3 - Evident use of phrases such as idioms, collocations, and lexical bundles but without much variety; some noticeable repetitions and misuses. 

4 - Appropriate use of a variety of phrases, such as idioms, collocations, and lexical bundles; occasional inaccuracies and colloquialisms. 

5 - Flexible and effective use of a variety of phrases, such as idioms, collocations, and lexical bundles, to convey precise and subtle meanings; rare minor inaccuracies that are negligible.
'''

syntax_rubric = '''
1 - Pervasive and basic errors in sentence structure and word  order that cause confusion; basic sentences errors common.

2 - Some sentence variation used; many sentence structure problems.

3 - Simple, compound, and complex syntactic structures present although the range may be limited; some apparent errors in sentence formation, especially in more complex sentences. 

4 - Appropriate use of a variety of syntactic structures, such as simple, compound, and complex sentences; occasional errors or inappropriateness in sentence formation. 

5 - Flexible and effective use of a full range of syntactic structures including simple, compound, and complex sentences; There may be rare minor and negligible errors in sentence formation.
'''

vocabulary_rubric = '''
1 - Limited vocabulary often inappropriately used; limited control of word choice and word forms; little attempt to use topic-related terms.

2 - Narrow range of vocabulary to convey basic and elementary meaning; topic related terms used inappropriately; errors in word formation and word choice that may distort meanings.

3 - Minimally adequate range of vocabulary for the topic; no precise use of subtle word meanings; topic related terms only used occasionally; attempts to use less common vocabulary but with some inaccuracy.

4 - Sufficient range of vocabulary to allow flexibility and precision; appropriate use of topic-related terms and less common lexical items.

5 - Wide range of vocabulary flexibly and effectively used to convey precise meanings; skillful use of topic-related terms and less common words; rare negligible inaccuracies in word use. 
'''

DIMENSION_RUBRIC = {
    "Overall": overall_rubric,
    "Cohesion": cohesion_rubric, 
    "Conventions": conventions_rubric,
    "Grammar": grammar_rubric,
    "Phraseology": phraseology_rubric,
    "Syntax": syntax_rubric,
    "Vocabulary": vocabulary_rubric
}

# Prompt Construction

In [4]:
with open(f"../data/ellipse/ellipse_sample.json", "r") as file:
    sample_list = json.load(file)

sampled_prompt = list()
sampled_essay = list()
sampled_score = list()

for sample in sample_list:
    sampled_prompt.append(sample["prompt"])
    sampled_essay.append(sample["full_text"])
    sampled_score.append(sample[DIMENSION])

# Prompt Construction

In [5]:
with open("../prompts/ellipse/think_aloud/gemini/user_prompt.txt", "r") as file:
    user_prompt = file.read()

In [6]:
gemini_response = model.generate_content(user_prompt.format(DIMENSION, DIMENSION, DIMENSION, DIMENSION,
                                        DIMENSION, DIMENSION_RUBRIC[DIMENSION], sampled_prompt[0], sampled_essay[0], sampled_score[0],
                                        sampled_prompt[1], sampled_essay[1], sampled_score[1], sampled_prompt[2], sampled_essay[2], sampled_score[2],
                                        sampled_prompt[3], sampled_essay[3], sampled_score[3], sampled_prompt[4], sampled_essay[4], sampled_score[4],
                                        sampled_prompt[5], sampled_essay[5], sampled_score[5]))

gemini_response = ''.join(re.findall(r'\{[^}]*\}', gemini_response.text))

In [7]:
result = eval(gemini_response)

In [8]:
attributes = ""
for i, attribute in result.items():
    attributes += f"- {attribute}\n"

In [9]:
print(attributes)

- Check for the range and appropriateness of vocabulary used.  A limited vocabulary, often inappropriately used, suggests a score of 1.  Look for struggles with word choice and word forms, and a lack of topic-specific terminology.
- Consider whether the vocabulary primarily conveys basic meaning, with limited range.  Inappropriate use of topic-related terms and errors in word formation/choice that distort meaning point to a score of 2.
- Evaluate if the vocabulary is adequate but not precise.  Look for a lack of subtle word usage and infrequent use of topic-related terms.  Attempts to use less common vocabulary may be present, but with inaccuracies, suggesting a score of 3.
- Assess whether the vocabulary demonstrates flexibility and precision.  Appropriate use of topic-related terms and less common lexical items signify a score of 4.
- Look for a wide range of vocabulary used effectively and precisely. Skillful use of topic-related terms and less common words, with very few inaccuraci